## Fake NEWS Classifier 

In [1]:
import pandas as pd 

In [2]:
df=pd.read_csv('final_clean_dataset.csv')


In [3]:
df.shape

(70247, 4)

In [4]:
df=df.dropna()

In [5]:
df.head()

,id,title,text,label
0,0,law enforcement on high alert following threat...,no comment is expected from barack obama membe...,1
1,2,unbelievable obamas attorney general says most...,now most of the demonstrators gathered last ni...,1
2,3,bob raised hindu uses story of christian conve...,a dozen politically active pastors came here f...,0
3,4,satan russia unvelis an image of its terrifyin...,the rs sarmat missile dubbed satan will replac...,1
4,5,about time christian group sues amazon and spl...,all we can say on this one is it s about time ...,1


In [6]:
df.shape

(70247, 4)

In [7]:
# get the independent features 
X=df.drop('label',axis=1)
# get the dependent feature
Y=df['label']

In [8]:
X.shape

(70247, 3)

In [9]:
Y.shape

(70247,)

In [47]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense, Input

In [11]:
# vocabulary size 
voc_size=5000

## One Hot Representation 

In [12]:
messages=X.copy()

In [13]:
# we are resetting indexes so which value are being deleted to adjust the sequence of index 
messages.reset_index(inplace=True)

In [14]:
import nltk
import re 
from nltk.corpus import stopwords 

In [15]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
# data preprocessing 
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()
corpus=[]
for i in range(0,len(messages)):

    # substitute everything with blank only keep english words
    review=re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review=review.lower()
    review=review.split()
          # if word is not in stopwords then do the stemming 
    review=[ps.stem(word) for word in review if not word in stopwords.words('english')]
    review=' '.join(review)
    corpus.append(review)

In [17]:
corpus

['law enforc high alert follow threat cop white blacklivesmatt fyf terrorist video',
 'unbeliev obama attorney gener say charlott rioter peac protestersin home state north carolina video',
 'bob rais hindu use stori christian convers woo evangel potenti bid',
 'satan russia unv imag terrifi new supernuk western world take notic',
 'time christian group sue amazon splc design hate group',
 'dr ben carson target never audit spoke nation prayer breakfast',
 'sport bar owner ban nfl gameswil show true american sport id like speak rural america video',
 'latest pipelin leak underscor danger dakota access pipelin',
 'gop senat smack punchabl altright nazi internet',
 'may brexit offer would hurt cost eu citizen eu parliament',
 'schumer call trump appoint offici overse puerto rico relief',
 'watch hilari ad call question health age clinton crime famili boss',
 'chang expect espn polit agenda despit huge subscrib declin breitbart',
 'billionair odebrecht brazil scandal releas hous arrest',
 '

In [18]:
# we are converting this in the one hot representation 
onehot_repr=[one_hot(words,voc_size)for words in corpus]
onehot_repr

[[4524, 3575, 3090, 281, 628, 3301, 779, 1151, 1217, 2997, 1797, 2429],
 [472,
  3181,
  4555,
  3331,
  3934,
  4195,
  2576,
  1077,
  3112,
  353,
  2126,
  1604,
  2643,
  2429],
 [3411, 1550, 2103, 1941, 3392, 65, 2152, 4065, 1408, 163, 4910],
 [1909, 2667, 589, 3976, 4151, 2078, 4172, 1980, 3236, 12, 737],
 [3358, 65, 2811, 4800, 331, 4841, 393, 4803, 2811],
 [3400, 3637, 1801, 4439, 1673, 119, 2582, 2652, 2727, 4424],
 [4769,
  3197,
  4259,
  3684,
  1746,
  3647,
  3738,
  479,
  613,
  4769,
  894,
  4433,
  4671,
  917,
  3478,
  2429],
 [3071, 879, 2982, 2062, 4212, 4485, 2038, 879],
 [1540, 2471, 3351, 2039, 3880, 1721, 2114],
 [3041, 4234, 661, 4906, 2872, 1985, 3911, 364, 3911, 2000],
 [4516, 1314, 2430, 798, 1547, 3984, 343, 2570, 2174],
 [1729, 1909, 1030, 1314, 4500, 1830, 4389, 3184, 3373, 4823, 2591],
 [2553, 1811, 651, 4241, 2636, 4452, 225, 210, 284, 3932],
 [4775, 3779, 1517, 224, 4880, 1405, 332],
 [151, 3551, 575, 2453, 3559, 2717, 954, 3730, 3189],
 [374, 2717

In [19]:
# make the sentences of fixes size 
# adding pad
set_length=20 
embedded_docs=pad_sequences(onehot_repr, padding='pre',maxlen=set_length) 
print(embedded_docs)

[[   0    0    0 ... 2997 1797 2429]
 [   0    0    0 ... 1604 2643 2429]
 [   0    0    0 ... 1408  163 4910]
 ...
 [   0    0    0 ... 4680   25 1235]
 [   0    0    0 ... 3294 2660 4790]
 [   0    0    0 ... 2760 3184 2946]]


In [20]:
len(embedded_docs)
embedded_docs[0]

array([   0,    0,    0,    0,    0,    0,    0,    0, 4524, 3575, 3090,
        281,  628, 3301,  779, 1151, 1217, 2997, 1797, 2429], dtype=int32)

In [48]:
# creating a model 
embedding_vector_features=50
model=Sequential([
    Input(shape=(set_length,)), 
    Embedding(voc_size,embedding_vector_features),
    LSTM(100),
    Dense(1,activation='sigmoid')
              ])

model.compile(loss='binary_crossentropy', optimizer='adam',metrics=['accuracy'])
print(model.summary())

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)              │ (None, 20, 50)              │         250,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_4 (LSTM)                        │ (None, 100)                 │          60,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 1)                   │             101 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 310,501 (1.18 MB)

 Trainable params: 310,501 (1.18 MB)

 Non-trainable params: 0 (0.00 B)

None


In [49]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(Y)

In [50]:
X_final.shape,y_final.shape

((70247, 20), (70247,))

In [51]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train,Y_test=train_test_split(X_final,y_final, test_size=0.33,random_state=42)

In [52]:
# finally training
model.fit(X_train,Y_train, validation_data=(X_test,Y_test),epochs=10,batch_size=64)


Epoch 1/10
736/736 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.8521 - loss: 0.3265 - val_accuracy: 0.8886 - val_loss: 0.2629
Epoch 2/10
736/736 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.9066 - loss: 0.2294 - val_accuracy: 0.8919 - val_loss: 0.2647
Epoch 3/10
736/736 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.9231 - loss: 0.1942 - val_accuracy: 0.8903 - val_loss: 0.2792
Epoch 4/10
736/736 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.9322 - loss: 0.1688 - val_accuracy: 0.8897 - val_loss: 0.2912
Epoch 5/10
736/736 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.9432 - loss: 0.1453 - val_accuracy: 0.8879 - val_loss: 0.3174
Epoch 6/10
736/736 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.9507 - loss: 0.1256 - val_accuracy: 0.8852 - val_loss: 0.3504
Epoch 7/10
736/736 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.9589 - loss: 0.1059 - val_accuracy: 0.8850 - val_loss: 0.3984
Epoch 8/10
736/736 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.9655 - loss: 0.0872 - 

In [53]:
y_pred=model.predict(X_test)

725/725 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


In [66]:
y_pred_binary=y_pred_binary.reshape(-1)
y_pred_binary=(y_pred>=0.5).astype(int)


In [64]:
from sklearn.metrics import confusion_matrix

In [67]:
confusion_matrix(Y_test,y_pred_binary)

array([[ 9920,  1574],
       [ 1141, 10547]])

In [68]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)


In [70]:
accuracy = accuracy_score(Y_test, y_pred_binary)
print("Accuracy:", accuracy)

Accuracy: 0.8828832715037529


In [74]:
precision=precision_score(Y_test,y_pred_binary)
print("precision:" ,precision)

precision: 0.8701427274977312


In [75]:
recall = recall_score(Y_test, y_pred_binary)
print("Recall:", recall)


Recall: 0.902378507871321


In [76]:
f1 = f1_score(Y_test, y_pred_binary)
print("F1 Score:", f1)


F1 Score: 0.8859674912848082


In [77]:
roc_auc = roc_auc_score(Y_test, y_pred)
print("ROC AUC:", roc_auc)


ROC AUC: 0.9381968825028729
